<a href="https://colab.research.google.com/github/adrian-lo-te/Sprints/blob/main/S7_N3_E2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from collections import defaultdict

La funció llegir_dades() rep la ruta del fitxer, l'obre i llegeix totes les línies. Utilitzo .strip() per eliminar els espais o salts de línia en blanc al principi i al final, filtrant només les línies que contenen informació.

In [ ]:
def llegir_dades(ruta):
    with open(ruta, 'r', encoding='utf-8') as text:
        return [linia.strip() for linia in text if linia.strip()]

La funció estadistiques() rep la llista de línies i calcula les mètriques de cada equip. Primer he creat un defaultdict amb l'estructura inicial de les mètriques.
Amb un bucle for recorro cada línia, utilitzant .split('\t') per separar els dos equips i el marcador. Després faig un .split('-') del marcador i converteixo els gols a enters amb map(int, ...). A continuació, actualitzo els gols i comprovo si ha guanyat, empatat o perdut cada equip per assignar-li els punts i el resultat corresponent.

In [ ]:
def estadistiques(linies):

    stats = defaultdict(lambda: {'Punts': 0, 'Victories': 0, 'Empats': 0, 'Derrotes': 0, 'GF': 0, 'GC': 0})

    for linia in linies:
        parts = linia.split('\t')
        if len(parts) != 3:
            continue
        eq1, marcador, eq2 = parts
        gols1, gols2 = map(int, marcador.split('-'))


        stats[eq1]['GF'] += gols1
        stats[eq1]['GC'] += gols2
        if gols1 > gols2:
            stats[eq1]['Victories'] += 1
            stats[eq1]['Punts'] += 3
        elif gols1 == gols2:
            stats[eq1]['Empats'] += 1
            stats[eq1]['Punts'] += 1
        else:
            stats[eq1]['Derrotes'] += 1


        stats[eq2]['GF'] += gols2
        stats[eq2]['GC'] += gols1
        if gols2 > gols1:
            stats[eq2]['Victories'] += 1
            stats[eq2]['Punts'] += 3
        elif gols2 == gols1:
            stats[eq2]['Empats'] += 1
            stats[eq2]['Punts'] += 1
        else:
            stats[eq2]['Derrotes'] += 1

    return stats

Aquesta funció transforma el diccionari d'estadístiques en un data frame. Primer converteixo les dades acumulades en una llista de diccionaris per a cada equip. Després he creat el data frame i ordeno la taula de forma descendent per la columna Punts fent servir .sort_values(). Finalment, reajusto l'índex per fer que la classificació comenci al número 1.

In [ ]:
def generar_classificacio(stats):

    llista = []
    for equip, s in stats.items():
        llista.append({
            'Equip': equip,
            'Punts': s['Punts'],
            'Victories': s['Victories'],
            'Empats': s['Empats'],
            'Derrotes': s['Derrotes'],
            'Gols a favor': s['GF'],
            'Gols en contra': s['GC']
        })
    classificacio = pd.DataFrame(llista)
    classificacio_final = classificacio.sort_values(by='Punts', ascending=False).reset_index(drop=True)
    classificacio_final.index = classificacio_final.index + 1
    return classificacio_final

La funció respostes() busca quins han estat els equips més destacats utilitzant .idxmax() per trobar l'índex del valor màxim de gols a favor i en contra, i .loc[] per extreure'n els noms i valors.

In [ ]:
def respostes(df):

    idx_golejador = df['Gols a favor'].idxmax()
    eq_golejador = df.loc[idx_golejador, 'Equip']
    g_favor = df.loc[idx_golejador, 'Gols a favor']

    idx_golejat = df['Gols en contra'].idxmax()
    eq_golejat = df.loc[idx_golejat, 'Equip']
    g_contra = df.loc[idx_golejat, 'Gols en contra']


    print(f"Equip més golejador: {eq_golejador} ({g_favor} gols)")
    print(f"Equip més golejat: {eq_golejat} ({g_contra} gols en contra)")
    print("\nClasificació:")
    display(df)

Per finalitzar, aquesta funció integra totes les anteriors per executar el flux complet de treball: llegeix el fitxer, calcula estadístiques, genera la classificació i mostra els resultats.

In [ ]:
def analisis_futbol(ruta):
    linies = llegir_dades(ruta)
    stats = estadistiques(linies)
    clasificacio = generar_classificacio(stats)
    respostes(clasificacio)
    return clasificacio

In [ ]:
arxiu = "/content/historic partits.txt"

resultat = analisis_futbol(arxiu)

Equip més golejador: Figueres (161 gols)
Equip més golejat: Vilafranca (172 gols en contra)

Clasificació:


,Equip,Punts,Victories,Empats,Derrotes,Gols a favor,Gols en contra
1,Girona FC,96,31,3,13,139,94
2,Llagostera,94,29,7,20,159,142
3,Sabadell,85,26,7,15,141,121
4,Cornellà,82,25,7,22,147,146
5,RCD Espanyol,80,23,11,21,131,144
6,Figueres,79,23,10,23,161,154
7,Lleida Esportiu,75,23,6,23,129,133
8,Terrassa,74,20,14,23,147,164
9,FC Barcelona,73,22,7,15,125,115
10,Vilafranca,72,20,12,25,157,172
